#CSV Ingestion via Delta Live Tables (Chunk 2)

In [0]:
# DLT Pipeline Script: 04_bronze_dlt_IDEMPOTENT
import dlt
from pyspark.sql.functions import current_timestamp, lit

@dlt.table(
    name="listings_csv_dlt",
    comment="Bronze: Idempotent Ingestion of Chunk 2 CSV",
    table_properties={
        "quality": "bronze",
        "delta.enableChangeDataFeed": "true"
    }
)
def listings_csv_dlt():
    return (
        # DLT automatically manages checkpoints, ensuring idempotency
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        # 1. FIXED: Specifically target Chunk 2 to avoid metadata overhead
        .option("pathGlobFilter", "1_main_chunk_2.csv") 
        .option("cloudFiles.inferColumnTypes", "false") # Keep as string for Silver safety
        .load("/Volumes/vstone_catalog/raw/chunks/")
        .withColumn("load_dt", current_timestamp())
        .withColumn("source_file", lit("1_main_chunk_2.csv"))
    )